# FOODIQ — category-specific quality grading models

This notebook trains **separate supervised classifiers** for milk, fermented products, fruit storage and beverages. It expects a CSV with actual measurements and an `expert_grade` label (`A`, `B`, or `C`) assigned using a documented evaluation SOP.

**Do not train on fabricated records or use a model prediction as the source of its own training label.** The model is decision support only. Mandatory FSSAI checks remain a separate rule-based layer in the Streamlit app.

In [ ]:
!pip -q install pandas scikit-learn joblib

## 1. Upload your labelled dataset

Use `sample_training_template.csv` from the starter project as a column guide. One row represents one tested sample. Keep the product category values exactly as `milk`, `fermented`, `fruit_storage`, or `beverages`.

In [ ]:
from google.colab import files
import io, os, zipfile, joblib
import numpy as np
import pandas as pd

uploaded = files.upload()


In [ ]:
# Load the first uploaded CSV file.
csv_names = [name for name in uploaded if name.lower().endswith('.csv')]
if not csv_names:
    raise ValueError('Upload a .csv dataset file.')
df = pd.read_csv(io.BytesIO(uploaded[csv_names[0]]))
print('Dataset shape:', df.shape)
display(df.head())
print('Columns:', df.columns.tolist())

## 2. Define the measurements for each model

These are measurements that the app can collect. Only use features that you actually measure reliably. Empty fields are imputed during training, but too many missing values will weaken the model.

In [ ]:
FEATURES = {
    'milk': ['product_type','temperature_c','humidity_pct','mq2_raw','mq135_raw',
             'milk_fat_pct','milk_snf_pct','milk_acidity_pct','milk_ph','density',
             'appearance_score','odour_score'],
    'fermented': ['product_type','temperature_c','humidity_pct','mq2_raw','mq135_raw',
                  'ph','titratable_acidity_pct','fat_pct','total_solids_pct',
                  'fermentation_hours','syneresis_score','appearance_score','odour_score'],
    'fruit_storage': ['product_type','temperature_c','humidity_pct','mq2_raw','mq135_raw',
                      'firmness_n','tss_brix','weight_loss_pct','ripeness_score','bruising_score',
                      'appearance_score','odour_score'],
    'beverages': ['product_type','temperature_c','humidity_pct','mq2_raw','mq135_raw',
                  'ph','tss_brix','titratable_acidity_pct','fruit_content_pct',
                  'appearance_score','odour_score','package_integrity'],
}
EXPECTED_CATEGORIES = list(FEATURES.keys())
required = {'category','product_type','expert_grade'}
missing = required - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {sorted(missing)}')

df['category'] = df['category'].astype(str).str.strip().str.lower()
df['expert_grade'] = df['expert_grade'].astype(str).str.strip().str.upper()
df = df[df['expert_grade'].isin(['A','B','C'])].copy()
print(df.groupby(['category','expert_grade']).size().unstack(fill_value=0))

## 3. Train and evaluate one model per category

The notebook uses a stratified train/test split, median imputation for numeric features, most-frequent imputation and one-hot encoding for `product_type`, and a Random Forest classifier. Inspect each class's precision, recall and F1-score rather than relying only on overall accuracy.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, balanced_accuracy_score
import matplotlib.pyplot as plt

os.makedirs('models', exist_ok=True)
trained = {}

for category, features in FEATURES.items():
    subset = df[df['category'] == category].copy()
    if subset.empty:
        print(f'\nSKIP {category}: no labelled rows')
        continue
    counts = subset['expert_grade'].value_counts()
    print(f'\n===== {category.upper()} =====')
    print('Rows:', len(subset), '| Class counts:', counts.to_dict())
    if subset['expert_grade'].nunique() < 2 or counts.min() < 2 or len(subset) < 12:
        print('SKIPPED: collect more independently labelled samples, including at least two examples per grade.')
        continue

    # Add absent feature columns so the notebook stays compatible with the Streamlit app.
    for col in features:
        if col not in subset.columns:
            subset[col] = np.nan
    X = subset[features].copy()
    y = subset['expert_grade'].copy()
    categorical = [c for c in features if c in ['product_type','package_integrity']]
    numeric = [c for c in features if c not in categorical]

    preprocessor = ColumnTransformer(transformers=[
        ('numeric', Pipeline([('imputer', SimpleImputer(strategy='median'))]), numeric),
        ('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),
                                  ('onehot', OneHotEncoder(handle_unknown='ignore'))]), categorical)
    ])
    model = Pipeline([
        ('preprocess', preprocessor),
        ('classifier', RandomForestClassifier(n_estimators=300, min_samples_leaf=2,
                                               class_weight='balanced', random_state=42))
    ])

    try:
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.25, random_state=42, stratify=y)
    except ValueError as exc:
        print('SKIPPED: unable to create stratified split:', exc)
        continue

    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    print('Balanced accuracy:', round(balanced_accuracy_score(y_test, pred), 3))
    print(classification_report(y_test, pred, labels=['A','B','C'], zero_division=0))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, labels=['A','B','C'],
                                            display_labels=['A','B','C'], cmap=None)
    plt.title(f'{category}: held-out test confusion matrix')
    plt.show()

    artifact = {'pipeline': model, 'features': features, 'category': category}
    joblib.dump(artifact, f'models/{category}_model.joblib')
    trained[category] = artifact
    print('Saved:', f'models/{category}_model.joblib')

if not trained:
    raise RuntimeError('No model trained. Check category values and collect more labelled data.')


## 4. Package the trained models

Download the zip, extract its `models/` directory, and copy that directory into the same folder as `app.py` in the Streamlit project.

In [ ]:
zip_name = 'foodiq_models.zip'
with zipfile.ZipFile(zip_name, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, filenames in os.walk('models'):
        for filename in filenames:
            full = os.path.join(root, filename)
            zf.write(full, arcname=full)
files.download(zip_name)

## 5. Model validation checklist

- Confirm the grade labels were assigned by a trained evaluator using the same written SOP.
- Use independent batches, collection dates and preferably multiple production lots in the hold-out set; random sample splitting can overestimate performance when many samples come from the same batch.
- Review precision, recall and confusion matrix for each grade. A low score for Grade C or a high rate of A predictions on poor-quality samples is not acceptable.
- Keep a separate final validation dataset that is not used for model selection.
- Re-train and re-validate whenever instruments, methods, product formulations or grading criteria change.
- A high model probability is not a probability of regulatory compliance or food safety.